How to Calculate Support and Resistance Levels Using Python: A Step-by-Step Guide

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import MetaTrader5 as mt5


In [2]:
from datetime import datetime, timezone

if not mt5.initialize():
    raise RuntimeError(mt5.last_error())

symbols = mt5.symbols_get()
print(f"Total symbols: {len(symbols)}")
for s in symbols[:50]:  # first 50
    print(s.name)

# filter JPY/metals
watch = [s.name for s in symbols if any(k in s.name for k in ["JPY", "XAU", "XAG"])]
print("\nFiltered:", watch[:100])

# --- MT5 vs system time (MT5 = last tick time on a liquid symbol; aligns with broker feed clock)
sys_utc = datetime.now(timezone.utc)
sys_local = datetime.now().astimezone()

tick = None
tick_symbol = None
for s in symbols[:50]:
    if mt5.symbol_select(s.name, True):
        tick = mt5.symbol_info_tick(s.name)
        if tick is not None:
            tick_symbol = s.name
            break

print("\n--- Time comparison ---")
print(f"System now (UTC):   {sys_utc.isoformat()}")
print(f"System now (local): {sys_local.isoformat()}")
if tick is None:
    print("MT5 tick time:      (no tick — could not read)")
else:
    mt5_utc = datetime.fromtimestamp(int(tick.time), tz=timezone.utc)
    skew_s = (mt5_utc - sys_utc).total_seconds()
    print(f"MT5 tick time (UTC) via {tick_symbol}: {mt5_utc.isoformat()}")
    print(f"Skew (MT5 tick UTC − system UTC): {skew_s:+.1f} s  (large gaps often mean quiet market or clock drift)")

_ = mt5.shutdown()

Total symbols: 1101
EURCZK.i
EURHUF.i
EURTRY.i
EURZAR.i
GBPTRY.i
GBPZAR.i
USDCZK.i
USDHUF.i
USDMXN.i
USDTRY.i
USDZAR.i
AUDUSD.i
EURUSD.i
GBPUSD.i
NZDUSD.i
USDCAD.i
USDCHF.i
USDJPY.i
AUDCAD.i
AUDCHF.i
AUDJPY.i
AUDNZD.i
CADCHF.i
CADJPY.i
CHFJPY.i
EURAUD.i
EURCAD.i
EURCHF.i
EURGBP.i
EURJPY.i
EURNOK.i
EURNZD.i
EURSEK.i
GBPAUD.i
GBPCAD.i
GBPCHF.i
GBPJPY.i
GBPNZD.i
NZDCAD.i
NZDCHF.i
NZDJPY.i
USDDKK.i
USDNOK.i
USDSEK.i
EURPLN.i
USDRUB.i
AUS200.i
DE30.i
ES35.i
F40.i

Filtered: ['USDJPY.i', 'AUDJPY.i', 'CADJPY.i', 'CHFJPY.i', 'EURJPY.i', 'GBPJPY.i', 'NZDJPY.i', 'XAGEUR.i', 'XAGUSD.i', 'XAUEUR.i', 'XAUUSD.i', 'USDJPY']

--- Time comparison ---
System now (UTC):   2026-05-10T09:13:58.946650+00:00
System now (local): 2026-05-10T12:43:58.946693+03:30
MT5 tick time (UTC) via EURCZK.i: 2026-05-08T23:54:59+00:00
Skew (MT5 tick UTC − system UTC): -119939.9 s  (large gaps often mean quiet market or clock drift)


In [3]:
from pathlib import Path
from datetime import datetime, timezone

SYMBOLS = [ 'BTCUSD'
# 'TRXUSD', 'DOGEUSD','LNKUSD','UNIUSD', 'LTCUSD','ADAUSD', 'AVEUSD', 'AVXUSD'
# 'BTCUSD', 'ETHUSD', 'SOLUSD', 'XRPUSD', 'BCHUSD', 'BNBUSD', 'EURUSD', 'NZDJPY', 'CADJPY', 'XAGUSD',
# 'XAUUSD', "USDJPY", "USDCAD", "USDCHF", 'AUDUSD', "EURCZK", "EURHUF", "EURTRY", "EURZAR",
# "GBPTRY", "GBPZAR", "USDCZK", "USDHUF", "USDMXN", "USDTRY", "USDZAR", "AUDUSD",
# "GBPUSD", "NZDUSD", "AUDCAD", 
# "AUDCHF", "AUDJPY", "AUDNZD", "CADCHF", "CADJPY",
]
TIMEFRAMES = ['D1', 'H1', 'H4', 'M15', 'M30', 'M5', 'M1']

# Optional date window (set datetime values or keep None)
DATE_FROM = None
DATE_TO = None
BARS = 5000000
BARS_BY_TF = {
    'M1': BARS,
    'M5': BARS,
    'M15': BARS,
    'M30': BARS,
    'H1': 100000,
    'H4': 90000,
    'D1': 3000,
}

MT5_TF = {
    'M1': mt5.TIMEFRAME_M1,
    'M5': mt5.TIMEFRAME_M5,
    'M15': mt5.TIMEFRAME_M15,
    'M30': mt5.TIMEFRAME_M30,
    'H1': mt5.TIMEFRAME_H1,
    'H4': mt5.TIMEFRAME_H4,
    'D1': mt5.TIMEFRAME_D1,
}


def fetch_mt5(symbol: str, tf: str, bars: int, date_from=None, date_to=None) -> pd.DataFrame:
    timeframe = MT5_TF[tf]

    if date_from is not None and date_to is not None:
        rates = mt5.copy_rates_range(symbol, timeframe, date_from, date_to)
    else:
        rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, bars)

    if rates is None or len(rates) == 0:
        last_error = mt5.last_error()
        raise RuntimeError(f'No data from MT5 for {symbol} {tf}. last_error={last_error}')

    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s', utc=True)
    df['symbol'] = symbol
    df['timeframe'] = tf
    return df


def fetch_mt5_with_retry(symbol: str, tf: str, bars: int, date_from=None, date_to=None) -> tuple[pd.DataFrame, int]:
    trial_bars = bars
    last_exc = None
    # Retry with progressively smaller history depth for higher timeframes
    for _ in range(4):
        try:
            return fetch_mt5(symbol, tf, bars=trial_bars, date_from=date_from, date_to=date_to), trial_bars
        except Exception as exc:
            last_exc = exc
            trial_bars = max(300, trial_bars // 2)
    raise RuntimeError(str(last_exc))


def resolve_symbol(base_symbol: str) -> str | None:
    # Try broker naming variants; many brokers use suffixes like .i
    candidates = [base_symbol, f'{base_symbol}.i']
    for candidate in candidates:
        if mt5.symbol_select(candidate, True):
            return candidate
    return None


if not mt5.initialize():
    raise RuntimeError(f'MT5 initialize() failed: {mt5.last_error()}')

base_dir = Path('./data')
results = {}
resolved_symbols = {}

try:
    for symbol in SYMBOLS:
        resolved = resolve_symbol(symbol)
        if resolved is None:
            print(f'Skip {symbol}: not available (tried {symbol} and {symbol}.i)')
            continue

        resolved_symbols[symbol] = resolved
        print(f'Using {symbol} -> {resolved}')

        for tf in TIMEFRAMES:
            tf_bars = BARS_BY_TF.get(tf, BARS)
            try:
                raw, used_bars = fetch_mt5_with_retry(
                    resolved,
                    tf,
                    bars=tf_bars,
                    date_from=DATE_FROM,
                    date_to=DATE_TO,
                )
                out_dir = base_dir / symbol / tf
                out_dir.mkdir(parents=True, exist_ok=True)
                out_file = out_dir / 'ohlcv.csv'
                raw.to_csv(out_file, index=False)
                results[(symbol, tf)] = len(raw)
                print(f'Cached {symbol} ({resolved}) {tf}: {len(raw)} rows (bars={used_bars}) -> {out_file}')
            except Exception as e:
                print(f'Failed {symbol} ({resolved}) {tf} (bars={tf_bars}): {e}')

    # Preview one cached result if available
    if results:
        preview_symbol, preview_tf = next(iter(results.keys()))
        preview_resolved = resolved_symbols[preview_symbol]
        preview_bars = BARS_BY_TF.get(preview_tf, BARS)
        preview, _ = fetch_mt5_with_retry(preview_resolved, preview_tf, bars=preview_bars, date_from=DATE_FROM, date_to=DATE_TO)
        preview.tail()
finally:
    mt5.shutdown()

Using BTCUSD -> BTCUSD.i
Cached BTCUSD (BTCUSD.i) D1: 3000 rows (bars=3000) -> data\BTCUSD\D1\ohlcv.csv
Cached BTCUSD (BTCUSD.i) H1: 68295 rows (bars=100000) -> data\BTCUSD\H1\ohlcv.csv
Cached BTCUSD (BTCUSD.i) H4: 17787 rows (bars=90000) -> data\BTCUSD\H4\ohlcv.csv
Cached BTCUSD (BTCUSD.i) M15: 262738 rows (bars=5000000) -> data\BTCUSD\M15\ohlcv.csv
Cached BTCUSD (BTCUSD.i) M30: 133145 rows (bars=5000000) -> data\BTCUSD\M30\ohlcv.csv
Cached BTCUSD (BTCUSD.i) M5: 778984 rows (bars=5000000) -> data\BTCUSD\M5\ohlcv.csv
Cached BTCUSD (BTCUSD.i) M1: 3809006 rows (bars=5000000) -> data\BTCUSD\M1\ohlcv.csv
